# Digital Audio Foundations: Building and Verifying a 440 Hz Sine Wave

This experiment maps sample indices to physical time, converts time into cycle progress, generates sine-wave amplitudes, and verifies that the resulting tensor represents the intended frequency.

In [ ]:
import torch

## 1. Audio configuration

The sample rate is the number of measurements per second. Frequency is the number of wave cycles per second. Duration determines the total number of samples.

In [ ]:
sample_rate = 22_050
frequency = 440
duration = 1

## 2. Sample indices

A one-second recording contains `sample_rate * duration` measurements. Indices start at zero, so the final index is `waveform_length - 1`. `float32` preserves these indices and supports the later time and sine calculations.

In [ ]:
waveform_length = sample_rate * duration
sample_indices = torch.arange(waveform_length, dtype=torch.float32)

In [ ]:
print("Waveform length:", waveform_length)
print("Sample-index shape:", sample_indices.shape)
print("Sample-index dtype:", sample_indices.dtype)
print("First sample index:", sample_indices[0].item())
print("Final three sample indices:", sample_indices[-3:])

## 3. Map indices to time

Each timestamp is calculated with `time = sample_index / sample_rate`. The final sample occurs slightly before one second because index `22050` is not included.

In [ ]:
time = sample_indices / sample_rate

In [ ]:
print("Time shape:", time.shape)
print("First time:", time[0].item())
print("Time at index 11025:", time[11025].item())
print("Final time:", time[-1].item())

## 4. Calculate cycle progress

`cycles_elapsed = frequency * time`. At half a second, a `440 Hz` wave has completed `220` cycles.

In [ ]:
cycles_elapsed = frequency * time

In [ ]:
print("Cycles-elapsed shape:", cycles_elapsed.shape)
print("Cycles at start:", cycles_elapsed[0].item())
print("Cycles at 0.5 seconds:", cycles_elapsed[11025].item())
print("Cycles at final sample:", cycles_elapsed[-1].item())

## 5. Convert cycle progress to amplitude

One complete cycle is `2*pi` radians, so the sine-wave formula is `sin(2*pi*cycles_elapsed)`. Omitting the factor of `2` would generate approximately `220 Hz` while still passing shape and range checks.

In [ ]:
amplitude = torch.sin(2 * torch.pi * cycles_elapsed)

In [ ]:
print("Amplitude shape:", amplitude.shape)
print("Amplitude at start:", amplitude[0].item())
print("Amplitude at 0.5 seconds:", amplitude[11025].item())
print("Minimum amplitude:", amplitude.min().item())
print("Maximum amplitude:", amplitude.max().item())

## 6. Add the batch dimension

The amplitude vector has shape `[T]`. `unsqueeze(0)` inserts a batch dimension and produces the waveform shape `[B, T] = [1, 22050]`.

In [ ]:
waveform = amplitude.unsqueeze(0)
samples_per_cycle = sample_rate / frequency

print("Waveform shape:", waveform.shape)
print("Samples per cycle:", samples_per_cycle)

## 7. Structural and numerical checks

These assertions verify shape, dtype, and amplitude range. They cannot prove the frequency by themselves.

In [ ]:
assert waveform.shape == (1, waveform_length)
assert waveform.dtype == torch.float32
assert torch.all(waveform >= -1.0)
assert torch.all(waveform <= 1.0)

## 8. Verify frequency with upward zero crossings

An upward zero crossing occurs when one sample is negative and the next sample is zero or positive. Comparing `amplitude[:-1]` with `amplitude[1:]` examines every adjacent pair. A sine wave has approximately one upward crossing per cycle.

The buffer begins exactly at zero and ends just before one second, so a `440 Hz` signal normally contains `439` counted negative-to-nonnegative transitions. A tolerance of one crossing accounts for this boundary convention.

In [ ]:
current_samples = amplitude[:-1]
next_samples = amplitude[1:]
upward_crossings = (current_samples < 0) & (next_samples >= 0)
upward_crossing_count = upward_crossings.sum().item()
expected_cycles = frequency * duration

print("Upward zero crossings:", upward_crossing_count)
print("Expected cycles in duration:", expected_cycles)

In [ ]:
assert abs(upward_crossing_count - expected_cycles) <= 1
print("All Chapter 02 checks passed.")

## Conclusion

The generated tensor is a one-second, batched, `float32`, `440 Hz` sine wave with amplitudes in `[-1, 1]`. Chapter 02 is complete. The next chapter applies the STFT and checks whether spectral energy appears near `440 Hz`.